# Forschungsfrage 3 - Formatierungsfehler: Moderner Non-LLM-Ansatz (XGBoost-Formatklassifikation + Parser)

Für `price` und `saving` wird zunächst per XGBoost das Formatmuster der Roh-Zeichenkette
**gelernt klassifiziert** (vgl. Abschnitt 2.2.1.3 / 3.3.3, statt wie im vorigen Notebook
durch handkodierte Regex-Bedingungen), anschließend wird derselbe zur Klasse passende
regelbasierte Parser angewendet, um den numerischen Wert zu extrahieren - die
Extraktionslogik selbst ist damit identisch zum traditionellen Ansatz; verglichen
wird ausschließlich der Formatklassifikations-Schritt (gelernt vs. handkodiert).

Für die Datumsspalten ist eine Formatklassifikation nicht sinnvoll, da das Rohformat
strukturell durchgängig einem einzigen Muster folgt (vgl. `01_Benchmark_...`,
Abschnitt 3) - hierfür siehe das Regelbasiert-Notebook, dessen Datums-Ergebnisse
auch für diese Methode gelten (kein separates ML-Modell erforderlich/sinnvoll).

**Wichtige methodische Erkenntnis (bitte in Abschnitt 3.2.1/4.3.2 der Arbeit
übernehmen):** Die bereinigte `saving`-Spalte ist **kein** einfach neu formatierter
Wert, sondern ein **abgeleitetes Verhältnis**:

```
saving_ratio = ersparnis_betrag / (bereinigter_preis + ersparnis_betrag)
```

(Ausnahme: Ist die Ersparnis bereits als Prozentsatz angegeben, z.B. `"50%"`, gilt
direkt `saving_ratio = prozent / 100`.) Diese Formel wurde durch Rückrechnung aus dem
Referenzdatensatz bestätigt (exakte Übereinstimmung auf mehreren Stichproben). Der
Rohwert `price` entspricht dabei dem bereits **rabattierten** Verkaufspreis; die
Summe aus Preis und Ersparnis ergibt den ursprünglichen (nicht rabattierten) Preis.

Damit die Bereinigung von `saving` realistisch bleibt, wird hier **kaskadiert**
gearbeitet: Der in diesem Notebook selbst für `price` erzeugte bereinigte Wert wird
als Preis-Kontext für die `saving`-Berechnung verwendet (kein Zugriff auf die
Ground-Truth-Preise) - Fehler in der Preisbereinigung wirken sich dadurch realistisch
auf die Ersparnisbereinigung aus.


## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import time
import os

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

SEED = 42
os.makedirs("results", exist_ok=True)

df_raw = pd.read_csv("data/rfd_main.csv").drop(columns=["Unnamed: 0"]).reset_index(drop=True)
df_raw["row_id"] = df_raw.index


## 2. Gemeinsame Hilfsfunktionen: String-Features und Formatklassifikation

In [2]:
def string_features(s):
    s = str(s)
    return {
        "length": len(s),
        "has_dollar": int("$" in s),
        "has_percent": int("%" in s),
        "has_alpha": int(bool(re.search(r"[A-Za-z]", s))),
        "has_slash": int("/" in s),
        "has_dash": int("-" in s),
        "n_digit_groups": len(re.findall(r"\d+", s)),
        "starts_with_digit": int(s[:1].isdigit()),
        "has_off_word": int("off" in s.lower()),
    }

def train_format_classifier(train_df, test_df):
    feat_cols = ["length", "has_dollar", "has_percent", "has_alpha", "has_slash",
                 "has_dash", "n_digit_groups", "starts_with_digit", "has_off_word"]
    X_train = pd.DataFrame([string_features(v) for v in train_df["raw_value"]])[feat_cols]
    X_test = pd.DataFrame([string_features(v) for v in test_df["raw_value"]])[feat_cols]

    le = LabelEncoder()
    y_train = le.fit_transform(train_df["format_class"])
    y_test_true = test_df["format_class"].values

    clf = xgb.XGBClassifier(n_estimators=150, max_depth=4, random_state=SEED, eval_metric="mlogloss")
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred = le.inverse_transform(clf.predict(X_test))
    class_acc = accuracy_score(y_test_true, y_pred)
    return y_pred, class_acc, train_time

def extract_first_number(s):
    m = re.search(r"(\d+(?:\.\d+)?)", str(s))
    return float(m.group(1)) if m else np.nan


## 3. `price`: Formatklassifikation + Parser

In [3]:
price_bench = pd.read_csv("benchmark/tf3_format_price.csv")
price_train = price_bench[price_bench["split"] == "train"].reset_index(drop=True)
price_test = price_bench[price_bench["split"] == "test"].reset_index(drop=True)

price_test = price_test.copy()
price_test["predicted_class"], price_class_acc, price_train_time = train_format_classifier(price_train, price_test)
print(f"Formatklassen-Erkennung (price) - Accuracy: {price_class_acc:.3f} (Trainingszeit {price_train_time:.2f}s)")

def parse_price_by_class(raw_value, format_class):
    s = str(raw_value)
    num = extract_first_number(s)
    if pd.isna(num):
        return np.nan
    if format_class == "prozent":
        # Ungewöhnlicher Randfall (Preis als Prozentangabe) - nicht sinnvoll interpretierbar
        return np.nan
    if format_class == "preisspanne":
        nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
        return float(np.mean(nums)) if nums else np.nan
    if format_class == "wortwert":
        return 0.0 if "free" in s.lower() else np.nan  # "varies" bleibt unbestimmt
    return num  # numerisch, dollar_prefix, mengenangabe, waehrungssuffix, sonstiges (best effort)

def predict_price(raw_value, classifier=None, feat_row=None):
    """Wiederverwendbare Pipeline: Formatklasse -> Parser. Nutzt für neue Werte den
    trainierten Klassifikator; hier direkt mit bereits vorhergesagter Klasse aufgerufen."""
    pass  # siehe price_test["predicted_class"] und parse_price_by_class oben

price_test["price_pred"] = [parse_price_by_class(r, c) for r, c in zip(price_test["raw_value"], price_test["predicted_class"])]

exact_match = (price_test["price_pred"].sub(price_test["true_clean_value"]).abs() < 0.01).mean()
valid_rate = price_test["price_pred"].notna().mean()
print(f"price - Exact-Match-Rate: {exact_match:.3f}  Valid-Format-Rate: {valid_rate:.3f}  (n_test={len(price_test)})")

price_test.to_csv("results/tf3_xgb_price_predictions.csv", index=False)


Formatklassen-Erkennung (price) - Accuracy: 0.981 (Trainingszeit 0.26s)
price - Exact-Match-Rate: 0.996  Valid-Format-Rate: 0.996  (n_test=257)


## 4. `saving`: Kaskadierte Preis-Bereinigung + Formatklassifikation + Ratio-Berechnung

In [4]:
saving_bench = pd.read_csv("benchmark/tf3_format_saving.csv")
saving_train = saving_bench[saving_bench["split"] == "train"].reset_index(drop=True)
saving_test = saving_bench[saving_bench["split"] == "test"].reset_index(drop=True)

# --- Rohen Preis für dieselben Zeilen holen und mit DEMSELBEN Klassifikator/Parser bereinigen ---
raw_price_lookup = df_raw.set_index("row_id")["price"]

def own_price_pipeline(raw_price_values):
    tmp = pd.DataFrame({"raw_value": raw_price_values})
    feat_cols = ["length", "has_dollar", "has_percent", "has_alpha", "has_slash",
                 "has_dash", "n_digit_groups", "starts_with_digit", "has_off_word"]
    X = pd.DataFrame([string_features(v) for v in tmp["raw_value"]])[feat_cols]
    # Formatklassifikator aus Abschnitt 3 wiederverwenden (auf price_train trainiert)
    X_train_full = pd.DataFrame([string_features(v) for v in price_train["raw_value"]])[feat_cols]
    le_local = LabelEncoder()
    y_train_full = le_local.fit_transform(price_train["format_class"])
    clf_local = xgb.XGBClassifier(n_estimators=150, max_depth=4, random_state=SEED, eval_metric="mlogloss")
    clf_local.fit(X_train_full, y_train_full)
    pred_classes = le_local.inverse_transform(clf_local.predict(X))
    return pd.Series([parse_price_by_class(r, c) for r, c in zip(tmp["raw_value"], pred_classes)], index=raw_price_values.index)

saving_test = saving_test.copy()
raw_prices_for_saving = raw_price_lookup.loc[saving_test["row_id_raw"]].reset_index(drop=True)
saving_test["own_cleaned_price"] = own_price_pipeline(raw_prices_for_saving).values

saving_test["predicted_class"], saving_class_acc, saving_train_time = train_format_classifier(saving_train, saving_test)
print(f"Formatklassen-Erkennung (saving) - Accuracy: {saving_class_acc:.3f} (Trainingszeit {saving_train_time:.2f}s)")

def parse_saving_ratio(raw_value, format_class, cleaned_price):
    amount = extract_first_number(raw_value)
    if pd.isna(amount):
        return np.nan
    if format_class == "prozent":
        return amount / 100.0
    if pd.isna(cleaned_price):
        return np.nan  # Preis konnte nicht bereinigt werden -> Ratio nicht berechenbar
    denom = cleaned_price + amount
    return amount / denom if denom > 0 else np.nan

saving_test["saving_pred"] = [
    parse_saving_ratio(r, c, p) for r, c, p in
    zip(saving_test["raw_value"], saving_test["predicted_class"], saving_test["own_cleaned_price"])
]

exact_match_saving = (saving_test["saving_pred"].sub(saving_test["true_clean_value"]).abs() < 0.01).mean()
valid_rate_saving = saving_test["saving_pred"].notna().mean()
print(f"saving - Exact-Match-Rate: {exact_match_saving:.3f}  Valid-Format-Rate: {valid_rate_saving:.3f}  (n_test={len(saving_test)})")

saving_test.to_csv("results/tf3_xgb_saving_predictions.csv", index=False)


Formatklassen-Erkennung (saving) - Accuracy: 0.987 (Trainingszeit 0.08s)
saving - Exact-Match-Rate: 0.946  Valid-Format-Rate: 0.980  (n_test=149)


## 5. Metriken und Laufzeit-Log speichern

In [5]:
metrics = {
    "experiment": "TF3_Formatierung", "method": "XGBoost_Parser",
    "price_exact_match": exact_match, "price_valid_rate": valid_rate, "price_class_accuracy": price_class_acc,
    "saving_exact_match": exact_match_saving, "saving_valid_rate": valid_rate_saving, "saving_class_accuracy": saving_class_acc,
    "hinweis": "Datumsspalten hier nicht enthalten - siehe Regelbasiert-Notebook (keine Formatklassifikation sinnvoll/noetig).",
}
with open("results/tf3_xgb_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

log_rows = pd.DataFrame([
    {"experiment": "TF3_Formatierung_price", "method": "XGBoost_Parser", "n_items": len(price_test),
     "wall_time_sec": price_train_time, "input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0, "model_name": "xgboost_parser"},
    {"experiment": "TF3_Formatierung_saving", "method": "XGBoost_Parser", "n_items": len(saving_test),
     "wall_time_sec": saving_train_time, "input_tokens": 0, "output_tokens": 0, "estimated_cost_usd": 0.0, "model_name": "xgboost_parser"},
])
log_path = "results/laufzeit_kosten_log.csv"
if os.path.exists(log_path):
    _old_log = pd.read_csv(log_path)
    _new_keys = set(zip(log_rows["experiment"], log_rows["method"]))
    _old_log = _old_log[~_old_log.apply(lambda r: (r["experiment"], r["method"]) in _new_keys, axis=1)]
    _combined_log = pd.concat([_old_log, log_rows], ignore_index=True)
else:
    _combined_log = log_rows
_combined_log.to_csv(log_path, index=False)

print("Gespeichert: results/tf3_xgb_{price,saving}_predictions.csv, results/tf3_xgb_metrics.json")


Gespeichert: results/tf3_xgb_{price,saving}_predictions.csv, results/tf3_xgb_metrics.json
